# RecoMart Streaming Ingestion
## Continuous User Interaction Log Processing

This notebook uses **Spark Structured Streaming with Auto Loader** to continuously ingest user interaction events.

**Architecture:**
* **Source**: `/Volumes/workspace/recomart_source_data/raw/streaming/user_interactions/`
* **Destination**: `recomart_bronze.user_interaction_logs` (Delta table)
* **Technology**: Auto Loader (cloudFiles) for incremental file discovery
* **Mode**: Append-only streaming writes

**Features:**
* Automatic schema inference and evolution
* Exactly-once processing semantics
* Checkpoint management for fault tolerance
* Real-time metrics and monitoring

**Usage:**
* Run all cells to start the streaming ingestion
* Monitor the dashboard for throughput and latency
* Stop execution to pause streaming (state is preserved in checkpoint)

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Configuration
# ══════════════════════════════════════════════════════════════════════════════

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from datetime import datetime

# ── PATHS ─────────────────────────────────────────────────────────────────
STREAM_SOURCE_PATH = "/Volumes/workspace/recomart_source_data/raw/streaming/user_interactions"
CHECKPOINT_PATH = "/Volumes/workspace/recomart_source_data/raw/checkpoints/user_interactions_stream"  # Fixed: inside raw volume
TARGET_TABLE = "recomart_bronze.user_interaction_logs"

# ── SCHEMA ────────────────────────────────────────────────────────────────
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), False),
    StructField("item_id", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("timestamp", StringType(), False),
    StructField("device", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("utm_source", StringType(), True)
])

print("✓ Configuration loaded")
print(f"  Source: {STREAM_SOURCE_PATH}")
print(f"  Checkpoint: {CHECKPOINT_PATH}")
print(f"  Target table: {TARGET_TABLE}")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Initialize Database
# ══════════════════════════════════════════════════════════════════════════════

# Ensure database exists
spark.sql("CREATE DATABASE IF NOT EXISTS recomart_bronze")
print("✓ Database recomart_bronze ready")

# Ensure checkpoint directory exists
try:
    dbutils.fs.mkdirs(CHECKPOINT_PATH)
    print(f"✓ Checkpoint directory ready")
except Exception as e:
    print(f"  Checkpoint: {e}")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Logging Infrastructure
# ══════════════════════════════════════════════════════════════════════════════

import uuid
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, LongType

# Create logging table if it doesn't exist (shared with batch ingestion)
spark.sql("""
    CREATE TABLE IF NOT EXISTS recomart_bronze.ingestion_logs (
        log_id STRING,
        run_id STRING,
        job_run_id BIGINT,
        notebook_name STRING,
        job_type STRING,
        status STRING,
        start_time TIMESTAMP,
        end_time TIMESTAMP,
        duration_seconds INT,
        records_processed INT,
        tables_updated STRING,
        error_message STRING,
        error_type STRING
    )
    USING DELTA
    COMMENT 'Execution logs for batch and streaming ingestion pipelines'
""")

# Create log file directory
LOG_FILE_PATH = "/Volumes/workspace/recomart_source_data/raw/logs"
try:
    dbutils.fs.mkdirs(LOG_FILE_PATH)
except:
    pass  # Directory might already exist

class IngestionLogger:
    """Logger for tracking ingestion pipeline execution"""
    
    def __init__(self, notebook_name: str, job_type: str, job_run_id: int = None):
        self.notebook_name = notebook_name
        self.job_type = job_type
        self.run_id = str(uuid.uuid4())[:8]
        self.start_time = datetime.now()
        self.records_processed = 0
        self.tables_updated = []
        
        # Use provided job_run_id or try to get from widget, fallback to None
        if job_run_id is not None:
            self.job_run_id = job_run_id
        else:
            try:
                self.job_run_id = int(dbutils.widgets.get("job_run_id"))
            except:
                self.job_run_id = None  # Running interactively
        
        print(f"\n{'='*80}")
        print(f"INGESTION RUN STARTED")
        print(f"{'='*80}")
        print(f"  Run ID: {self.run_id}")
        if self.job_run_id:
            print(f"  Job Run ID: {self.job_run_id}")
        print(f"  Notebook: {self.notebook_name}")
        print(f"  Job Type: {self.job_type}")
        print(f"  Start Time: {self.start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*80}\n")
        
        # Log start
        self._write_log('RUNNING', None, None, None, None)
    
    def add_records(self, count: int):
        """Increment the record count (safely handles None)"""
        self.records_processed += (count or 0)
    
    def add_table(self, table_name: str):
        """Add a table to the updated list"""
        self.tables_updated.append(table_name)
    
    def log_success(self):
        """Log successful completion"""
        end_time = datetime.now()
        duration = int((end_time - self.start_time).total_seconds())
        
        print(f"\n{'='*80}")
        print(f"✅ INGESTION COMPLETED SUCCESSFULLY")
        print(f"{'='*80}")
        print(f"  Run ID: {self.run_id}")
        if self.job_run_id:
            print(f"  Job Run ID: {self.job_run_id}")
        print(f"  Duration: {duration} seconds")
        print(f"  Records Processed: {self.records_processed:,}")
        print(f"  Tables Updated: {', '.join(self.tables_updated)}")
        print(f"  End Time: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*80}\n")
        
        self._write_log('SUCCESS', end_time, duration, None, None)
    
    def log_failure(self, error: Exception):
        """Log failed execution"""
        end_time = datetime.now()
        duration = int((end_time - self.start_time).total_seconds())
        error_type = type(error).__name__
        error_message = str(error)
        
        print(f"\n{'='*80}")
        print(f"❌ INGESTION FAILED")
        print(f"{'='*80}")
        print(f"  Run ID: {self.run_id}")
        if self.job_run_id:
            print(f"  Job Run ID: {self.job_run_id}")
        print(f"  Duration: {duration} seconds")
        print(f"  Records Processed: {self.records_processed:,}")
        print(f"  Error Type: {error_type}")
        print(f"  Error Message: {error_message}")
        print(f"  End Time: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*80}\n")
        
        self._write_log('FAILED', end_time, duration, error_message, error_type)
    
    def _write_log(self, status: str, end_time, duration, error_message, error_type):
        """Write log entry to Delta table AND text file"""
        log_schema = StructType([
            StructField("log_id", StringType(), False),
            StructField("run_id", StringType(), False),
            StructField("job_run_id", LongType(), True),
            StructField("notebook_name", StringType(), False),
            StructField("job_type", StringType(), False),
            StructField("status", StringType(), False),
            StructField("start_time", TimestampType(), False),
            StructField("end_time", TimestampType(), True),
            StructField("duration_seconds", IntegerType(), True),
            StructField("records_processed", IntegerType(), True),
            StructField("tables_updated", StringType(), True),
            StructField("error_message", StringType(), True),
            StructField("error_type", StringType(), True)
        ])
        
        log_data = [(
            str(uuid.uuid4()),
            self.run_id,
            self.job_run_id,
            self.notebook_name,
            self.job_type,
            status,
            self.start_time,
            end_time,
            duration,
            self.records_processed if self.records_processed > 0 else None,
            ','.join(self.tables_updated) if self.tables_updated else None,
            error_message,
            error_type
        )]
        
        log_df = spark.createDataFrame(log_data, log_schema)
        
        # 1. Write to Delta table
        log_df.write.format("delta").mode("append").saveAsTable("recomart_bronze.ingestion_logs")
        
        # 2. Write to text file
        try:
            # Create log file name with timestamp
            timestamp = self.start_time.strftime("%Y%m%d_%H%M%S")
            log_file = f"{LOG_FILE_PATH}/{self.job_type.lower()}_{timestamp}_{self.run_id}.log"
            
            # Format log entry as human-readable text
            log_text = f"""{'='*80}
INGESTION LOG ENTRY
{'='*80}
Run ID: {self.run_id}
Job Run ID: {self.job_run_id or 'N/A (interactive)'}
Notebook: {self.notebook_name}
Job Type: {self.job_type}
Status: {status}
Start Time: {self.start_time.strftime('%Y-%m-%d %H:%M:%S')}
End Time: {end_time.strftime('%Y-%m-%d %H:%M:%S') if end_time else 'N/A'}
Duration: {duration} seconds" if duration else 'N/A'
Records Processed: {self.records_processed if self.records_processed > 0 else 0}
Tables Updated: {', '.join(self.tables_updated) if self.tables_updated else 'N/A'}
Error Type: {error_type or 'N/A'}
Error Message: {error_message or 'N/A'}
{'='*80}
"""
            
            # Write to file using dbutils
            dbutils.fs.put(log_file, log_text, overwrite=True)
            
        except Exception as e:
            # Don't fail the whole logging if text file write fails
            print(f"  ⚠️  Warning: Could not write text log file: {e}")

print("✓ Logging infrastructure initialized")
print("  Log table: recomart_bronze.ingestion_logs")
print(f"  Log files: {LOG_FILE_PATH}")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Auto Loader Streaming Query (Serverless Compatible)
# ══════════════════════════════════════════════════════════════════════════════

# Initialize logger with job_run_id from widget
try:
    job_run_id = int(dbutils.widgets.get("job_run_id"))
except:
    job_run_id = None
logger = IngestionLogger(notebook_name="task2_streaming_ingestion", job_type="STREAMING", job_run_id=job_run_id)

try:
    print("\n" + "="*80)
    print("STARTING STREAMING INGESTION (SERVERLESS MODE)")
    print("="*80)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("\nNOTE: Serverless uses 'availableNow' trigger - processes all available")
    print("      files then stops. Schedule this notebook to run frequently for")
    print("      continuous ingestion (e.g., every 1-2 minutes).")
    print("\nReading from streaming source with Auto Loader...")
    print("-" * 80)

    # Read stream using Auto Loader (cloudFiles)
    df_stream = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", f"{CHECKPOINT_PATH}/schema")
            .option("header", "true")
            .schema(schema)
            .load(STREAM_SOURCE_PATH)
    )

    # Add ingestion metadata (Unity Catalog compatible)
    df_enriched = (
        df_stream
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_type", F.lit("streaming_csv"))
            .withColumn("_batch_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss"))
            .withColumn("_source_path", F.col("_metadata.file_path"))  # Unity Catalog compatible
    )

    print("✓ Stream source configured with Auto Loader")
    print("✓ Metadata columns added")
    print("\nStarting write stream to Bronze table...\n")

    # Write stream to Delta table (availableNow for serverless)
    query = (
        df_enriched.writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", CHECKPOINT_PATH)
            .option("mergeSchema", "true")
            .trigger(availableNow=True)  # Serverless compatible: process all available then stop
            .toTable(TARGET_TABLE)
    )

    # CRITICAL FIX: Wait for the streaming query to complete
    query.awaitTermination()

    # Get total record count from all micro-batches in recentProgress
    total_records = 0
    if query.recentProgress:
        for progress in query.recentProgress:
            if progress and 'numInputRows' in progress:
                # Safely handle None values - be explicit
                num_rows = progress.get('numInputRows')
                if num_rows is not None:
                    total_records += num_rows
        logger.add_records(total_records)
        logger.add_table("user_interaction_logs")
        print(f"✓ Streaming query completed successfully")
        print(f"  Query ID: {query.id}")
        print(f"  Micro-batches processed: {len(query.recentProgress)}")
        print(f"  Total records ingested: {total_records:,}")
    else:
        # No data processed
        logger.add_records(0)
        logger.add_table("user_interaction_logs")
        print(f"✓ Streaming query completed successfully")
        print(f"  Query ID: {query.id}")
        print(f"  Records ingested: 0 (no new files)")

    print("\n" + "="*80)
    print("✅ INGESTION COMPLETE (This Run)")
    print("="*80)
    print("\nAll available files have been processed.")
    print("\n🗓️  For continuous ingestion, schedule this notebook to run every 1-2 minutes.")
    print("    This gives you near-real-time data with serverless efficiency.\n")
    
    # Log success
    logger.log_success()
    
except Exception as e:
    # Log failure with error details
    logger.log_failure(e)
    print(f"\n❌ ERROR: {str(e)}")
    print("\nFull stack trace:")
    import traceback
    traceback.print_exc()
    raise  # Re-raise to stop notebook execution

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Monitor Streaming Metrics
# ══════════════════════════════════════════════════════════════════════════════

import time

print("\n" + "="*80)
print("STREAMING METRICS DASHBOARD")
print("="*80)
print(f"Monitoring stream: {query.name}")
print("-" * 80)

try:
    while query.isActive:
        # Get current status
        status = query.status
        progress = query.lastProgress
        
        if progress:
            # Extract metrics
            batch_id = progress.get('batchId', 'N/A')
            num_rows = progress.get('numInputRows', 0)
            input_rate = progress.get('inputRowsPerSecond', 0)
            process_rate = progress.get('processedRowsPerSecond', 0)
            
            # Timing info
            trigger_exec = progress.get('durationMs', {})
            latency_ms = trigger_exec.get('triggerExecution', 0)
            
            print(f"[Batch {batch_id}] "
                  f"Rows: {num_rows:4d} | "
                  f"Input: {input_rate:6.1f} r/s | "
                  f"Process: {process_rate:6.1f} r/s | "
                  f"Latency: {latency_ms:6.0f} ms | "
                  f"{datetime.now().strftime('%H:%M:%S')}")
        
        time.sleep(10)  # Update every 10 seconds
        
except KeyboardInterrupt:
    print("\n\n⚠️  Monitoring stopped by user")
    print("\nStreaming query is still active. Stop the previous cell to halt ingestion.")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Query Current Table Stats
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("TABLE STATISTICS")
print("="*80)

# Get table stats
df_table = spark.table(TARGET_TABLE)

total_rows = df_table.count()
unique_users = df_table.select("user_id").distinct().count()
unique_items = df_table.select("item_id").distinct().count()

print(f"\nTable: {TARGET_TABLE}")
print(f"  Total rows:      {total_rows:,}")
print(f"  Unique users:    {unique_users:,}")
print(f"  Unique items:    {unique_items:,}")

# Event type distribution
print(f"\nEvent type distribution:")
event_dist = df_table.groupBy("event_type").count().orderBy("count", ascending=False)
event_dist.show(truncate=False)

# Recent events
print("\nRecent events (last 10):")
df_table.orderBy(F.col("timestamp").desc()).select(
    "event_id", "user_id", "item_id", "event_type", "timestamp", "device"
).show(10, truncate=False)

print("="*80)

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# View Ingestion Execution Logs
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("INGESTION EXECUTION LOGS")
print("="*80)

# Query logs from the logging table
logs_df = spark.table("recomart_bronze.ingestion_logs").orderBy("start_time", ascending=False).limit(20)

print(f"\nShowing last 20 ingestion runs:\n")
display(logs_df.select(
    "run_id",
    "job_run_id",
    "notebook_name",
    "job_type",
    "status",
    "start_time",
    "duration_seconds",
    "records_processed",
    "tables_updated",
    "error_type"
))

print("\nLegend:")
print("  run_id: Internal short UUID for this execution")
print("  job_run_id: Databricks job run ID (NULL if interactive)")
print("="*80)